In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(".../data/aved_raw.csv", na_values=['', ' ', 'NULL', 'NaN', 'null'])
print(f"Original rows: {len(df)}")

time_col = 'time' 
if time_col not in df.columns:
    time_col = df.columns[0]
    print(f"Using '{time_col}' as time column.")

df[time_col] = pd.to_datetime(df[time_col], utc=True)
df.set_index(time_col, inplace=True)

orig_tz = df.index.tz
df.index = df.index.tz_localize(None)

df.index = df.index.floor('1T')

new_times = df.index.values.copy()
minutes = df.index.minute
even_mask = (minutes % 2 == 0)
new_times[even_mask] -= np.timedelta64(1, 'm')
df.index = pd.DatetimeIndex(new_times)

df_aligned = df.groupby(level=0).first()

df_aligned = df_aligned[df_aligned.index.minute % 2 != 0]

if orig_tz is not None:
    df_aligned.index = df_aligned.index.tz_localize(orig_tz)
df_aligned = df_aligned.sort_index()

print(f"--- Completed ---")
print(f"Aligned total row: {len(df_aligned)}")
print(f"Even timestamp exists: {(df_aligned.index.minute % 2 == 0).any()}")


def check_data_integrity(original_df, processed_df):
    stats = []
    for col in original_df.columns:
        if "value" in col:
            stats.append({
                'Feature': col,
                'Original_NaN': original_df[col].isna().sum(),
                'Original_Zeros': (original_df[col] == 0).sum(),
                'Aligned_NaN': processed_df[col].isna().sum() if col in processed_df else np.nan,
                'Aligned_Zeros': (processed_df[col] == 0).sum() if col in processed_df else np.nan
            })
    return pd.DataFrame(stats)

comparison = check_data_integrity(df, df_aligned)
print(comparison)

cols_to_show = [c for c in ['BIOLOGY.LINE 3 TANK 1.N2O value', 'BIOLOGY.LINE 3 TANK 1.NH4 value'] if c in df_aligned.columns]
print(df_aligned[cols_to_show].head())

<>:4: SyntaxWarning: invalid escape sequence '\C'
<>:4: SyntaxWarning: invalid escape sequence '\C'
C:\Users\lenovo\AppData\Local\Temp\ipykernel_28432\243328955.py:4: SyntaxWarning: invalid escape sequence '\C'
  df = pd.read_csv("E:\\2025-2026\CIVE70088\Coursework\\aved_raw.csv", na_values=['', ' ', 'NULL', 'NaN', 'null'])


Original rows: 906815


C:\Users\lenovo\AppData\Local\Temp\ipykernel_28432\243328955.py:18: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df.index = df.index.floor('1T')


--- Completed ---
Aligned total row: 526302
Even timestamp exists: False
                                          Feature  Original_NaN  \
0         BIOLOGY.BLOWERSTATION 1.Q.AIRFLOW value        386227   
1         BIOLOGY.LINE 3 TANK 1 VALVE 1.PCT value        403980   
2                 BIOLOGY.LINE 3 TANK 1.N2O value        386269   
3                 BIOLOGY.LINE 3 TANK 1.NH4 value        386220   
4                 BIOLOGY.LINE 3 TANK 1.NO3 value        386232   
5                  BIOLOGY.LINE 3 TANK 1.O2 value        386214   
6         BIOLOGY.LINE 3 TANK 1.O2.SETPOINT value        380530   
7        BIOLOGY.LINE 3 TANK 1.PROCESSPHASE value        387466   
8           BIOLOGY.LINE 3 TANK 1.Q.AIRFLOW value        436612   
9                  BIOLOGY.LINE 3 TANK 1.SS value        386226   
10        BIOLOGY.LINE 3 TANK 1.TEMPERATURE value        400208   
11        BIOLOGY.LINE 3 TANK 2 VALVE 1.PCT value        433449   
12                 BIOLOGY.LINE 3 TANK 2.O2 value       

In [18]:
df_aligned.to_csv("df_aligned3.csv")

In [19]:
print(df_aligned)

                           BIOLOGY.BLOWERSTATION 1.Q.AIRFLOW value  \
2022-06-11 22:01:00+00:00                             26829.375000   
2022-06-11 22:03:00+00:00                             29626.341797   
2022-06-11 22:05:00+00:00                             28605.208984   
2022-06-11 22:07:00+00:00                             27850.439453   
2022-06-11 22:09:00+00:00                             26726.554688   
...                                                            ...   
2024-06-11 21:51:00+00:00                             18526.916016   
2024-06-11 21:53:00+00:00                             20769.166016   
2024-06-11 21:55:00+00:00                             23267.166016   
2024-06-11 21:57:00+00:00                             24089.083984   
2024-06-11 21:59:00+00:00                                      NaN   

                           BIOLOGY.BLOWERSTATION 1.Q.AIRFLOW quality  \
2022-06-11 22:01:00+00:00                                        1.0   
2022-06-11 22:0